# Run Talent Agent with the configured defaults

Run the cells from top to bottom to start a normal Talent Agent search. The CLI reads `config/search.json`, loads credentials from the project `.env`, and uses the default candidate CV. `MODEL_OVERRIDE` in the run cell selects the model for this notebook; re-running resumes the most recent incomplete run when its saved listing count fits the configured `max_results`. If `max_results` is `null`, no local result cap is applied.

## Prerequisites

- Install the project dependencies in a Python environment and select that environment as this notebook's kernel (for example, `talent-agent/.venv`).
- Create `talent-agent/.env` with `OPENROUTER_API_KEY`, `APIFY_API_TOKEN`, and `APIFY_ACTOR_ID` configured. Do not put secrets in this notebook.
- Ensure the default candidate CV exists at `candidate/master_cv.docx`.
- Review `config/search.json` before running; search and model requests may incur charges or use API quota.

In [1]:
from pathlib import Path
import subprocess
import sys

# The notebook lives in the project's notebooks/ directory; also support launching it from the repository root.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError("Could not locate talent-agent/pyproject.toml from the notebook working directory")

print(f"Project: {PROJECT_ROOT}")
print(f"Python:  {sys.executable}")
assert (PROJECT_ROOT / "candidate" / "master_cv.docx").is_file(), "Default CV not found: candidate/master_cv.docx"
assert (PROJECT_ROOT / ".env").is_file(), "Project .env not found; configure credentials before running"


Project: /home/nadeem-bhatti/repos/talent-agent
Python:  /home/nadeem-bhatti/repos/talent-agent/.venv/bin/python


In [8]:
import json
from datetime import date, datetime

MODEL_OVERRIDE = "nvidia/nemotron-3-ultra-550b-a55b:free"

config_path = PROJECT_ROOT / "config" / "search.json"
search_config = json.loads(config_path.read_text(encoding="utf-8"))
output_settings = search_config["output"]
output_root = Path(output_settings["root_dir"])
if not output_root.is_absolute():
    output_root = PROJECT_ROOT / output_root
output_root.mkdir(parents=True, exist_ok=True)
max_results = search_config["max_results"]
if max_results is not None:
    max_results = int(max_results)
base_run_name = date.today().strftime(output_settings["date_format"])
if output_settings["dated_subfolders"]:
    run_directories = [
        path for path in output_root.iterdir()
        if path.is_dir() and (path.name == base_run_name or path.name.startswith(f"{base_run_name}_"))
    ]
else:
    run_directories = [output_root, *(path for path in output_root.iterdir() if path.is_dir())]

resume_directory = None
for candidate_run in sorted(run_directories, key=lambda path: path.stat().st_mtime, reverse=True):
    candidate_state_path = candidate_run / "agent_outputs" / "run_state.json"
    if not candidate_state_path.is_file():
        continue
    candidate_state = json.loads(candidate_state_path.read_text(encoding="utf-8"))
    statuses = candidate_state.get("stages", {})
    if not statuses or all(status == "complete" for status in statuses.values()):
        continue
    artifacts = candidate_state.get("artifacts", {})
    raw_filename = artifacts.get("raw_jobs_filename", output_settings["raw_jobs_filename"])
    raw_path = candidate_run / raw_filename
    if not raw_path.is_file():
        continue
    saved_jobs = json.loads(raw_path.read_text(encoding="utf-8"))
    if not isinstance(saved_jobs, list):
        continue
    if max_results is not None and len(saved_jobs) > max_results:
        print(
            f"Skipping incomplete run with {len(saved_jobs)} listings; "
            f"the configured total max_results is {max_results}."
        )
        continue
    resume_directory = candidate_run
    break

command = [sys.executable, "-m", "talent_agent.cli", "run"]
if resume_directory is not None:
    run_directory = resume_directory
    command.extend(["--resume", str(run_directory)])
    print(f"Resuming interrupted run: {run_directory}")
else:
    run_id = f"notebook-{datetime.now().strftime('%H%M%S')}"
    command.append("--search")
    command.extend(["--run-id", run_id])
    if output_settings["dated_subfolders"]:
        run_directory = output_root / f"{base_run_name}_{run_id}"
    else:
        run_directory = output_root / run_id
    print(
        f"Starting a new search (max_results={max_results if max_results is not None else 'unlimited'}): "
        f"{run_directory}"
    )

if MODEL_OVERRIDE:
    command.extend(["--model", MODEL_OVERRIDE])
    if resume_directory is not None:
        command.append("--allow-model-change-on-resume")

completed = subprocess.run(command, cwd=PROJECT_ROOT, text=True)
if completed.returncode:
    raise RuntimeError(
        f"Talent Agent stopped with exit status {completed.returncode}. "
        f"The run is checkpointed at {run_directory}; rerun this cell to resume it."
    )
print("Talent Agent finished successfully.")


Resuming interrupted run: /home/nadeem-bhatti/repos/talent-agent/outputs/2026-10-23_notebook-091506

🚀 Talent Agent is starting with nvidia/nemotron-3-ultra-550b-a55b:free.
📄 Candidate CV loaded from /home/nadeem-bhatti/repos/talent-agent/candidate/master_cv.docx.

⏯️ Resuming saved run in /home/nadeem-bhatti/repos/talent-agent/outputs/2026-10-23_notebook-091506…
   Found 100 unique job listing(s).
   🗂️ Raw listings saved to /home/nadeem-bhatti/repos/talent-agent/outputs/2026-10-23_notebook-091506/raw_jobs.json

🧭 Researcher restored 45 listing(s) from saved output.
   45 listing(s) passed the relevance review.
   8 listing(s) remain after employment, compensation, and security-clearance filters.

🧠 Career Analyst is scoring 8 listing(s) against your CV (minimum score 70)…
   🧠 Career Analyst restored batch 1/3 from saved results.
   🧠 Career Analyst restored batch 2/3 from saved results.
   🧠 Career Analyst restored batch 3/3 from saved results.
   ✅ 4 role(s) qualified.

💼 Applicati

In [4]:
from IPython.display import Markdown, display

if not run_directory.is_dir():
    print("No output run directory was found.")
else:
    print(f"Run: {run_directory}")
    for artifact in sorted(run_directory.rglob("*")):
        if artifact.is_file():
            relative_path = artifact.relative_to(PROJECT_ROOT)
            display(Markdown(f"- [{relative_path}]({artifact.resolve().as_uri()})"))


Run: /home/nadeem-bhatti/repos/talent-agent/outputs/2026-10-22_notebook-093258


- [outputs/2026-10-22_notebook-093258/agent_outputs/career_analyst.json](file:///home/nadeem-bhatti/repos/talent-agent/outputs/2026-10-22_notebook-093258/agent_outputs/career_analyst.json)

- [outputs/2026-10-22_notebook-093258/agent_outputs/cover_letter_writer.json](file:///home/nadeem-bhatti/repos/talent-agent/outputs/2026-10-22_notebook-093258/agent_outputs/cover_letter_writer.json)

- [outputs/2026-10-22_notebook-093258/agent_outputs/cv_editor.json](file:///home/nadeem-bhatti/repos/talent-agent/outputs/2026-10-22_notebook-093258/agent_outputs/cv_editor.json)

- [outputs/2026-10-22_notebook-093258/agent_outputs/document_reviewer.json](file:///home/nadeem-bhatti/repos/talent-agent/outputs/2026-10-22_notebook-093258/agent_outputs/document_reviewer.json)

...

- [outputs/2026-10-22_notebook-093258/applications/Paragon_Alpha_-_Hedge_Fund_Talent_Business_Systems_Engineer_-_Quant_Hedge_Fund_CoverLetter.docx](file:///home/nadeem-bhatti/repos/talent-agent/outputs/2026-10-22_notebook-093258/applications/Paragon_Alpha_-_Hedge_Fund_Talent_Business_Systems_Engineer_-_Quant_Hedge_Fund_CoverLetter.docx)

- [outputs/2026-10-22_notebook-093258/prioritized_jobs.json](file:///home/nadeem-bhatti/repos/talent-agent/outputs/2026-10-22_notebook-093258/prioritized_jobs.json)

- [outputs/2026-10-22_notebook-093258/raw_jobs.json](file:///home/nadeem-bhatti/repos/talent-agent/outputs/2026-10-22_notebook-093258/raw_jobs.json)

- [outputs/2026-10-22_notebook-093258/tracker.csv](file:///home/nadeem-bhatti/repos/talent-agent/outputs/2026-10-22_notebook-093258/tracker.csv)

- [outputs/2026-10-22_notebook-093258/tracker.html](file:///home/nadeem-bhatti/repos/talent-agent/outputs/2026-10-22_notebook-093258/tracker.html)